# Step 4 — PlanTUS: Interactive Transducer Placement

Runs PlanTUS to interactively place a transducer on the scalp surface. A SimNIBS
head mesh is converted to wb_view surfaces with placement metric maps; clicking a
scalp vertex triggers optimised placement computation. The result is exported as a
BrainSight-compatible coordinate file.

**Typical interaction time:** 5–15 min per session.

[← Step 3 — Inverse Registration](step03_inverse_registration.ipynb) &nbsp;|&nbsp; **Step 4** &nbsp;|&nbsp; Step 5 →

In [6]:
# Reference card — displays the pipeline overview and all option tables.
# Run this cell once to see settings options before editing the Settings cell below.
%run z_references/step4_reference.py

In [5]:
# -----------------------------------------------------------------------
# Settings — edit here before running
# -----------------------------------------------------------------------

SITE_YAML = "../config/sites/site_RIKEN_AK.yaml"
SUB_ID    = "sub-NS"

# ── Target ────────────────────────────────────────────────
TARGET_NAME       = "aMCC_NeuroSynthTopic112"   # target label (matches mask label from step 3)
TARGET_SIDE       = "_R"                         # "_R", "_L", or "" for bilateral
ADDITIONAL_OFFSET = 0.0                          # gel/pad offset (mm)

# ── Run options ───────────────────────────────────────────
DRY_RUN = False   # True = validate inputs only; no files written

# True  — use pynput mouse listener (requires macOS Accessibility permission)
# False — parse wb_view FINER log directly; no Accessibility permission needed
USE_PYNPUT = True

print("Settings loaded.")
print(f"  Subject:  {SUB_ID}")
print(f"  Target:   {TARGET_NAME}{TARGET_SIDE}")
print(f"  DRY_RUN:  {DRY_RUN}")


Settings loaded.
  Subject:  sub-NS
  Target:   aMCC_NeuroSynthTopic112_R
  DRY_RUN:  False


In [ ]:
# -----------------------------------------------------------------------
# Imports & path setup
# -----------------------------------------------------------------------

# ── Imports (stdlib) ──────────────────────────────────────
import sys
from pathlib import Path

# ── Config — Python path ──────────────────────────────────
# Make src/ importable; one level up from run/
SRC_DIR = str((Path("../src")).resolve())
if SRC_DIR not in sys.path:sys.path.insert(0, SRC_DIR)

# ── Paths — resolve relative to absolute ──────────────────
SITE_YAML = str(Path(SITE_YAML).resolve())

# ── src / utils ───────────────────────────────────────────
from utils import (
    load_site_config,
    resolve_data_dir,
    normalise_sub_id,
    load_transducer_config,
    transducer_params,
    setup_environment,
    run_plantus,
    write_brainsight_for_vtx,
)

print("Imports complete.")


Imports complete.


In [ ]:
# -----------------------------------------------------------------------
# Step 4-0 — Load config, resolve paths, set up environment
# Validates SimNIBS m2m directory; configures PATH; loads transducer params.
# -----------------------------------------------------------------------

cfg = load_site_config(SITE_YAML)

data_dir                 = resolve_data_dir(cfg)
sub_id_full, sub_id_bare = normalise_sub_id(SUB_ID)
m2m_dir                  = data_dir / sub_id_bare / f"m2m_{sub_id_full}"

if not m2m_dir.exists():
    raise FileNotFoundError(f"SimNIBS m2m directory not found: {m2m_dir}")

print(f"Subject:  {sub_id_full}")
print(f"Data dir: {data_dir}")
print(f"m2m_dir:  {m2m_dir}")

setup_environment(cfg)

tcfg = load_transducer_config(cfg, SITE_YAML)
tp   = transducer_params(tcfg)

print(f"\nTransducer:       {tcfg.get('name', 'unknown')}")
print(f"Focal depth (mm): {tp.get('focal_depth_mm')}")


Subject:  sub-NS
Data dir: /Users/atsushikikumoto/Dropbox/w_SCRIPTS/P.TUSPractice/SUB_LIST_PREP
m2m_dir:  /Users/atsushikikumoto/Dropbox/w_SCRIPTS/P.TUSPractice/SUB_LIST_PREP/NS/m2m_sub-NS


In [ ]:
# -----------------------------------------------------------------------
# Step 4b — PlanTUS interactive placement
# 1. prepscene: mesh → surfaces + metric maps + scene file → wb_view opens.
# 2. Click scalp vertex → "yes" runs placement; "no" repeats selection.
# DRY_RUN=True validates inputs only; no files written.
# -----------------------------------------------------------------------

run_plantus(
    sub_id_full=sub_id_full,
    sub_id_bare=sub_id_bare,
    m2m_dir=m2m_dir,
    target_name=TARGET_NAME,
    target_side=TARGET_SIDE,
    tp=tp,
    additional_offset=ADDITIONAL_OFFSET,
    dry_run=DRY_RUN,
    use_pynput=USE_PYNPUT,
)

### Step 4b-alt — Launch `PlanTUS_wrapper.py` as a subprocess

Use the cell below **instead of** Step 4b above when the notebook kernel is **not** `simnibs_python`.  
The wrapper receives all settings from the cells above — no duplication required.

In [ ]:
# -----------------------------------------------------------------------
# Step 4b-alt — Launch PlanTUS_wrapper.py as a subprocess
# All settings are forwarded from the Settings cell above.
# -----------------------------------------------------------------------
import subprocess

_simnibs_py  = cfg.get("simnibs_python", "simnibs_python")
_wrapper     = str((Path(SITE_YAML).parent.parent.parent / "PlanTUS" / "PlanTUS_wrapper.py").resolve())

_cmd = [
    str(_simnibs_py),
    _wrapper,
    "--site",              SITE_YAML,
    "--sub",               SUB_ID,
    "--target",            TARGET_NAME,
    "--side",              TARGET_SIDE,
    "--additional-offset", str(ADDITIONAL_OFFSET),
]
if DRY_RUN:
    _cmd.append("--dry-run")

print("Command:\n  " + " \\\n    ".join(_cmd))
subprocess.run(_cmd, check=True)

In [ ]:
# -----------------------------------------------------------------------
# Step 4c — Export BrainSight-compatible coordinate file
# Output: <m2m_dir>/<sub_id>_<target_name><target_side>_brainsight.txt
# -----------------------------------------------------------------------

write_brainsight_for_vtx(
    m2m_dir=m2m_dir,
    sub_id_full=sub_id_full,
    target_name=TARGET_NAME,
    target_side=TARGET_SIDE,
    coordinate_system=cfg.get("coordinate_system", "NIfTI:Scanner"),
)

Saved BrainSight file: /Users/atsushikikumoto/Dropbox/w_SCRIPTS/P.TUSPractice/SUB_LIST_PREP/NS/m2m_sub-NS/PlanTUS/sub-NS_T1w_aMCC_NeuroSynthTopic112_mask_native_R/sub-NS_T1w_aMCC_NeuroSynthTopic112_R_target_coordinates_Brainsight_PlanTUS.txt
  Target (Scanner): [ 4.81919851 29.3975728  15.35490199]
  Entry  (Scanner): [ 4.84813595 52.10019684 49.97498322]


PosixPath('/Users/atsushikikumoto/Dropbox/w_SCRIPTS/P.TUSPractice/SUB_LIST_PREP/NS/m2m_sub-NS/PlanTUS/sub-NS_T1w_aMCC_NeuroSynthTopic112_mask_native_R/sub-NS_T1w_aMCC_NeuroSynthTopic112_R_target_coordinates_Brainsight_PlanTUS.txt')

In [ ]:
# -----------------------------------------------------------------------
# Step 4d — Merge L+R BrainSight files into a single combined file
# -----------------------------------------------------------------------
#
# Run this cell after running Step 4c for BOTH _L and _R sides.
# Looks for individual per-side BrainSight files and merges them into:
#   <m2m_dir/PlanTUS/combined>/
#     {sub_id}_T1w_{TARGET_NAME}_target_coordinates_Brainsight_PlanTUS.txt
#
# If only one side has been run, prints a warning and skips.

from utils import find_plantus_target_folder, merge_brainsight_files

_side_paths = {}
for _side in ("_L", "_R"):
    try:
        _folder = find_plantus_target_folder(m2m_dir, sub_id_full, TARGET_NAME, _side)
        _label = f"{sub_id_full}_T1w_{TARGET_NAME}{_side}"
        _cand = _folder / f"{_label}_target_coordinates_Brainsight_PlanTUS.txt"
        if _cand.exists():
            _side_paths[_side] = _cand
    except SystemExit:
        pass  # folder doesn't exist yet

if len(_side_paths) < 2:
    print(f"WARNING: found {list(_side_paths.keys())} — both _L and _R required for merge.")
    print("Run Step 4c for the missing side first, then re-run this cell.")
else:
    _combined_dir = m2m_dir / "PlanTUS" / "combined"
    _combined_dir.mkdir(parents=True, exist_ok=True)
    _combined_path = _combined_dir / f"{sub_id_full}_T1w_{TARGET_NAME}_target_coordinates_Brainsight_PlanTUS.txt"
    merge_brainsight_files(
        in_paths=[_side_paths["_L"], _side_paths["_R"]],
        out_path=_combined_path,
    )
    print(f"  Contains L (_target + _entry) and R (_target + _entry) rows.")
    print(f"  Ready to import into BrainSight.")
